# Exploración inicial de los datos

In [1]:
import pandas as pd

PATH = "paysim/PS_20174392719_1491204439457_log.csv"

df = pd.read_csv(
    PATH,
    usecols=["step", "nameOrig", "isFraud"]
)

lengths = df.groupby("nameOrig").size()
sender_labels = df.groupby("nameOrig")["isFraud"].max()

print("Transacciones:", len(df))
print("Remitentes únicos:", df["nameOrig"].nunique())
print("\nDistribución de transacciones por remitente:")
print(lengths.describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]))

print("\nRemitentes por longitud:")
for minimum in [1, 2, 3, 5, 10, 20]:
    print(f"{minimum:2d} o más: {(lengths >= minimum).sum():,}")

fraud_senders = sender_labels[sender_labels == 1].index
fraud_lengths = lengths.loc[fraud_senders]

print("\nRemitentes fraudulentos:", len(fraud_senders))
print("Longitud de remitentes fraudulentos:")
print(fraud_lengths.describe())

print("\nFraudulentos por longitud:")
for minimum in [1, 2, 3, 5, 10]:
    print(f"{minimum:2d} o más: {(fraud_lengths >= minimum).sum():,}")

Transacciones: 6362620
Remitentes únicos: 6353307

Distribución de transacciones por remitente:
count    6.353307e+06
mean     1.001466e+00
std      3.832002e-02
min      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
90%      1.000000e+00
95%      1.000000e+00
99%      1.000000e+00
max      3.000000e+00
dtype: float64

Remitentes por longitud:
 1 o más: 6,353,307
 2 o más: 9,298
 3 o más: 15
 5 o más: 0
10 o más: 0
20 o más: 0

Remitentes fraudulentos: 8213
Longitud de remitentes fraudulentos:
count    8213.000000
mean        1.003409
std         0.058293
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         2.000000
dtype: float64

Fraudulentos por longitud:
 1 o más: 8,213
 2 o más: 28
 3 o más: 0
 5 o más: 0
10 o más: 0


99.85 % de los remitentes tiene una sola transacción.
Solo 9,313 remitentes tienen más de una.
La longitud máxima es 3.
Solo 28 remitentes fraudulentos tienen dos transacciones.
Ningún remitente fraudulento alcanza tres transacciones.

Con estos datos, una LSTM no podría aprender frecuencia, cambios de destino ni evolución temporal. Hacer padding sobre secuencias de longitud uno produciría una arquitectura secuencial solo de nombre.

In [2]:
from pathlib import Path
import pandas as pd

BASE = Path("ibm-transactions")

files = [
    BASE / "HI-Small_Trans.csv",
    BASE / "LI-Small_Trans.csv",
]

for path in files:
    sample = pd.read_csv(path, nrows=10)

    print("=" * 70)
    print("Archivo:", path.name)
    print(f"Tamaño: {path.stat().st_size / 1024**2:,.2f} MB")
    print("Columnas:")
    print(sample.columns.tolist())
    print("\nTipos inferidos:")
    print(sample.dtypes)
    print("\nPrimeras filas:")
    print(sample.head(10).to_string(index=False))

Archivo: HI-Small_Trans.csv
Tamaño: 453.63 MB
Columnas:
['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']

Tipos inferidos:
Timestamp                 str
From Bank               int64
Account                   str
To Bank                 int64
Account.1                 str
Amount Received       float64
Receiving Currency        str
Amount Paid           float64
Payment Currency          str
Payment Format            str
Is Laundering           int64
dtype: object

Primeras filas:
       Timestamp  From Bank   Account  To Bank Account.1  Amount Received Receiving Currency  Amount Paid Payment Currency Payment Format  Is Laundering
2022/09/01 00:20         10 8000EBD30       10 8000EBD30          3697.34          US Dollar      3697.34        US Dollar   Reinvestment              0
2022/09/01 00:20       3208 8000F4580        1 8000F5340             0.01          US 

In [3]:
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import time

BASE = Path("ibm-transactions")

FILES = [
    BASE / "HI-Small_Trans.csv",
    BASE / "LI-Small_Trans.csv",
]

USECOLS = [
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "Is Laundering",
]

DTYPES = {
    "From Bank": "int32",
    "Account": "string",
    "To Bank": "int32",
    "Account.1": "string",
    "Amount Received": "float64",
    "Receiving Currency": "category",
    "Amount Paid": "float64",
    "Payment Currency": "category",
    "Payment Format": "category",
    "Is Laundering": "int8",
}


def audit_ibm_file(path, chunksize=500_000):
    start = time.time()

    sender_stats = None
    format_counts = Counter()
    paid_currency_counts = Counter()
    received_currency_counts = Counter()

    total_rows = 0
    positive_rows = 0
    self_transfers = 0
    currency_changes = 0
    timestamp_min = None
    timestamp_max = None

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            path,
            usecols=USECOLS,
            dtype=DTYPES,
            chunksize=chunksize,
        ),
        start=1,
    ):
        total_rows += len(chunk)
        positive_rows += int(chunk["Is Laundering"].sum())

        timestamps = pd.to_datetime(
            chunk["Timestamp"],
            format="%Y/%m/%d %H:%M",
            errors="coerce",
        )

        current_min = timestamps.min()
        current_max = timestamps.max()

        if pd.notna(current_min):
            timestamp_min = (
                current_min
                if timestamp_min is None
                else min(timestamp_min, current_min)
            )

        if pd.notna(current_max):
            timestamp_max = (
                current_max
                if timestamp_max is None
                else max(timestamp_max, current_max)
            )

        same_account = (
            (chunk["From Bank"] == chunk["To Bank"])
            & (chunk["Account"] == chunk["Account.1"])
        )
        self_transfers += int(same_account.sum())

        different_currency = (
            chunk["Payment Currency"].astype("string")
            != chunk["Receiving Currency"].astype("string")
        )
        currency_changes += int(different_currency.sum())

        format_counts.update(
            chunk["Payment Format"]
            .astype("string")
            .value_counts()
            .to_dict()
        )

        paid_currency_counts.update(
            chunk["Payment Currency"]
            .astype("string")
            .value_counts()
            .to_dict()
        )

        received_currency_counts.update(
            chunk["Receiving Currency"]
            .astype("string")
            .value_counts()
            .to_dict()
        )

        # Estadísticas por remitente usando una clave compuesta.
        chunk_sender_stats = (
            chunk.groupby(
                ["From Bank", "Account"],
                sort=False,
                observed=True,
            )["Is Laundering"]
            .agg(n_transactions="size", is_positive="max")
        )

        if sender_stats is None:
            sender_stats = chunk_sender_stats
        else:
            sender_stats = (
                pd.concat([sender_stats, chunk_sender_stats])
                .groupby(level=[0, 1], sort=False)
                .agg(
                    n_transactions=("n_transactions", "sum"),
                    is_positive=("is_positive", "max"),
                )
            )

        print(
            f"{path.name} | chunk {chunk_number:02d} | "
            f"filas procesadas: {total_rows:,}"
        )

    lengths = sender_stats["n_transactions"]
    positive_sender_mask = sender_stats["is_positive"] == 1
    positive_lengths = sender_stats.loc[
        positive_sender_mask,
        "n_transactions",
    ]

    elapsed = time.time() - start

    print("\n" + "=" * 75)
    print("ARCHIVO:", path.name)
    print(f"Tamaño: {path.stat().st_size / 1024**2:,.2f} MB")
    print(f"Tiempo de auditoría: {elapsed / 60:.2f} minutos")
    print(f"Periodo: {timestamp_min} — {timestamp_max}")

    print("\nTRANSACCIONES")
    print(f"Total: {total_rows:,}")
    print(f"Positivas: {positive_rows:,}")
    print(f"Proporción positiva: {positive_rows / total_rows:.8%}")
    print(f"Transferencias a la misma cuenta: {self_transfers:,}")
    print(f"Cambios de moneda: {currency_changes:,}")

    print("\nREMITENTES")
    print(f"Remitentes únicos: {len(sender_stats):,}")
    print(f"Remitentes positivos: {positive_sender_mask.sum():,}")
    print(
        "Proporción de remitentes positivos: "
        f"{positive_sender_mask.mean():.8%}"
    )

    print("\nLONGITUD DE TODAS LAS SECUENCIAS")
    print(
        lengths.describe(
            percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
        )
    )

    print("\nREMITENTES POR LONGITUD MÍNIMA")
    for minimum in [1, 2, 3, 5, 10, 20, 30, 50, 100]:
        count = int((lengths >= minimum).sum())
        print(f"{minimum:3d} o más: {count:,}")

    print("\nLONGITUD DE SECUENCIAS POSITIVAS")
    if len(positive_lengths) > 0:
        print(
            positive_lengths.describe(
                percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
            )
        )

        print("\nREMITENTES POSITIVOS POR LONGITUD MÍNIMA")
        for minimum in [1, 2, 3, 5, 10, 20, 30, 50, 100]:
            count = int((positive_lengths >= minimum).sum())
            print(f"{minimum:3d} o más: {count:,}")
    else:
        print("No se encontraron remitentes positivos.")

    print("\nFORMATOS DE PAGO")
    for key, value in format_counts.most_common():
        print(f"{key}: {value:,}")

    print("\nMONEDAS DE PAGO")
    for key, value in paid_currency_counts.most_common():
        print(f"{key}: {value:,}")

    print("\nMONEDAS RECIBIDAS")
    for key, value in received_currency_counts.most_common():
        print(f"{key}: {value:,}")

    return {
        "file": path.name,
        "total_rows": total_rows,
        "positive_rows": positive_rows,
        "positive_transaction_rate": positive_rows / total_rows,
        "unique_senders": len(sender_stats),
        "positive_senders": int(positive_sender_mask.sum()),
        "positive_sender_rate": float(positive_sender_mask.mean()),
        "median_length": float(lengths.median()),
        "p95_length": float(lengths.quantile(0.95)),
        "p99_length": float(lengths.quantile(0.99)),
        "max_length": int(lengths.max()),
        "senders_length_5": int((lengths >= 5).sum()),
        "senders_length_10": int((lengths >= 10).sum()),
        "positive_senders_length_5": int(
            (positive_lengths >= 5).sum()
        ),
        "positive_senders_length_10": int(
            (positive_lengths >= 10).sum()
        ),
    }


summaries = []

for file_path in FILES:
    summaries.append(audit_ibm_file(file_path))

comparison = pd.DataFrame(summaries).set_index("file")

print("\n" + "=" * 75)
print("COMPARACIÓN FINAL")
display(comparison.T)

HI-Small_Trans.csv | chunk 01 | filas procesadas: 500,000
HI-Small_Trans.csv | chunk 02 | filas procesadas: 1,000,000
HI-Small_Trans.csv | chunk 03 | filas procesadas: 1,500,000
HI-Small_Trans.csv | chunk 04 | filas procesadas: 2,000,000
HI-Small_Trans.csv | chunk 05 | filas procesadas: 2,500,000
HI-Small_Trans.csv | chunk 06 | filas procesadas: 3,000,000
HI-Small_Trans.csv | chunk 07 | filas procesadas: 3,500,000
HI-Small_Trans.csv | chunk 08 | filas procesadas: 4,000,000
HI-Small_Trans.csv | chunk 09 | filas procesadas: 4,500,000
HI-Small_Trans.csv | chunk 10 | filas procesadas: 5,000,000
HI-Small_Trans.csv | chunk 11 | filas procesadas: 5,078,345

ARCHIVO: HI-Small_Trans.csv
Tamaño: 453.63 MB
Tiempo de auditoría: 0.23 minutos
Periodo: 2022-09-01 00:00:00 — 2022-09-18 16:18:00

TRANSACCIONES
Total: 5,078,345
Positivas: 5,177
Proporción positiva: 0.10194266%
Transferencias a la misma cuenta: 591,212
Cambios de moneda: 72,170

REMITENTES
Remitentes únicos: 496,999
Remitentes positivos:

file,HI-Small_Trans.csv,LI-Small_Trans.csv
total_rows,5.078345e+06,6.924049e+06
positive_rows,5.177000e+03,3.565000e+03
positive_transaction_rate,1.019427e-03,5.148722e-04
unique_senders,4.969990e+05,6.812830e+05
positive_senders,3.376000e+03,2.382000e+03
positive_sender_rate,6.792770e-03,3.496344e-03
median_length,2.000000e+00,2.000000e+00
p95_length,4.600000e+01,4.600000e+01
p99_length,9.200000e+01,9.200000e+01
max_length,1.686720e+05,2.220370e+05
